# Cálculo Numérico - Lista de Exercícios
**Sistemas lineares e implementação numérica** - Resolução

## Parte 1 - Conceitos e resolução manual
### Exercício 1 - Forma matricial $A\mathbf{x}=\mathbf{b}$
**a)** $A=\begin{bmatrix}2&1\\1&-3\end{bmatrix},\ \mathbf{x}=\begin{bmatrix}x\\y\end{bmatrix},\ \mathbf{b}=\begin{bmatrix}7\\-5\end{bmatrix}$

**b)** $A=\begin{bmatrix}1&2&-1\\3&-1&2\\2&1&1\end{bmatrix},\ \mathbf{x}=\begin{bmatrix}x\\y\\z\end{bmatrix},\ \mathbf{b}=\begin{bmatrix}2\\9\\7\end{bmatrix}$

### Exercício 2 - Eliminação de Gauss
Sistema: $2x+y-z=8,\ -3x-y+2z=-11,\ -2x+y+2z=-3$.

Matriz aumentada inicial:
$$\left[\begin{array}{ccc|c}2&1&-1&8\\-3&-1&2&-11\\-2&1&2&-3\end{array}\right]$$

Passo 1 (pivô 2): $L_2\leftarrow L_2+\tfrac32L_1$ e $L_3\leftarrow L_3+L_1$:
$$\left[\begin{array}{ccc|c}2&1&-1&8\\0&\tfrac12&\tfrac12&1\\0&2&1&5\end{array}\right]$$

Passo 2 (pivô $\tfrac12$): $L_3\leftarrow L_3-4L_2$:
$$\left[\begin{array}{ccc|c}2&1&-1&8\\0&\tfrac12&\tfrac12&1\\0&0&-1&1\end{array}\right]$$

Substituição regressiva: $-z=1\Rightarrow z=-1$; $\tfrac12y+\tfrac12(-1)=1\Rightarrow y=3$; $2x+3+1=8\Rightarrow x=2$.

**Resposta: $(x,y,z)=(2,3,-1)$.**

## Parte 2 - Propriedades e interpretação
### Exercício 3 - Número de soluções
**a)** $L_2\leftarrow L_2-2L_1$ dá $[0\ 0\,|\,0]$. Há 1 pivô para 2 incógnitas e nenhuma contradição: **infinitas soluções** ($y$ livre, $x=3-y$).

**b)** $L_2\leftarrow L_2-2L_1$ dá $[0\ 0\,|\,1]$, isto é, $0=1$: **nenhuma solução** (sistema inconsistente).

**c)** $L_2\leftarrow L_2-2L_1$ dá $[0\ -3\,|\,-6]$. Há 2 pivôs para 2 incógnitas: **solução única**, $y=2$ e $x=1$.

## Parte 3 - Implementação numérica em Python
### Exercício 4 - Eliminação de Gauss sem pivoteamento

In [1]:
import numpy as np

def gauss_sem_pivoteamento(A, b, verbose=True):
    """Resolve Ax = b por eliminação de Gauss sem pivoteamento."""
    A = np.array(A, dtype=float)
    b = np.array(b, dtype=float).reshape(-1, 1)
    n = A.shape[0]

    # Matriz aumentada [A | b]
    M = np.hstack([A, b])

    # Eliminação: zera os elementos abaixo de cada pivô
    for k in range(n - 1):
        if abs(M[k, k]) < 1e-12:
            raise ZeroDivisionError(f"Pivô nulo na posição ({k},{k}): "
                                    "seria necessário pivoteamento.")
        for i in range(k + 1, n):
            m = M[i, k] / M[k, k]            # multiplicador
            M[i, k:] = M[i, k:] - m * M[k, k:]
        if verbose:
            print(f"Matriz aumentada após passo {k + 1}:\n{M}\n")

    if abs(M[n - 1, n - 1]) < 1e-12:
        raise ZeroDivisionError("Pivô nulo: sistema singular.")

    # Substituição regressiva
    x = np.zeros(n)
    for i in range(n - 1, -1, -1):
        x[i] = (M[i, n] - M[i, i + 1:n] @ x[i + 1:n]) / M[i, i]
    return x

Teste com o sistema do exercício 2 e comparação com `numpy.linalg.solve`:

In [2]:
A2 = [[2, 1, -1],
      [-3, -1, 2],
      [-2, 1, 2]]
b2 = [8, -11, -3]

x_gauss = gauss_sem_pivoteamento(A2, b2)
x_numpy = np.linalg.solve(A2, b2)
print("Solução (Gauss implementado):", x_gauss)
print("Solução (numpy.linalg.solve):", x_numpy)
print("Diferença máxima:", np.max(np.abs(x_gauss - x_numpy)))

Matriz aumentada após passo 1:
[[ 2.   1.  -1.   8. ]
 [ 0.   0.5  0.5  1. ]
 [ 0.   2.   1.   5. ]]

Matriz aumentada após passo 2:
[[ 2.   1.  -1.   8. ]
 [ 0.   0.5  0.5  1. ]
 [ 0.   0.  -1.   1. ]]

Solução (Gauss implementado): [ 2.  3. -1.]
Solução (numpy.linalg.solve): [ 2.  3. -1.]
Diferença máxima: 1.2212453270876722e-15


**Interpretação:** o algoritmo reproduz $(2,3,-1)$, igual ao numpy, com diferença da ordem de $10^{-15}$ devida a arredondamento em ponto flutuante. Sem pivoteamento, o método falha se algum pivô for zero e perde precisão se for muito pequeno.

## Parte 4 - Aplicação
### Exercício 5 - Balanço térmico
**a) Resolução manual.**
$$\left[\begin{array}{ccc|c}4&-1&0&15\\-1&4&-1&10\\0&-1&3&10\end{array}\right]$$

$L_2\leftarrow L_2+\tfrac14L_1$ e $L_3\leftarrow L_3+\tfrac4{15}L_2$:
$$\left[\begin{array}{ccc|c}4&-1&0&15\\0&\tfrac{15}4&-1&\tfrac{55}4\\0&0&\tfrac{41}{15}&\tfrac{41}3\end{array}\right]$$

$T_3=\tfrac{41/3}{41/15}=5$; $\tfrac{15}4T_2-5=\tfrac{55}4\Rightarrow T_2=5$; $4T_1-5=15\Rightarrow T_1=5$.

**Resposta: $T_1=T_2=T_3=5$.** Verificação: $4(5)-5=15$; $-5+20-5=10$; $-5+15=10$.

**b) Implementação** (reutiliza a função do exercício 4):

In [3]:
A5 = [[4, -1, 0],
      [-1, 4, -1],
      [0, -1, 3]]
b5 = [15, 10, 10]

T_gauss = gauss_sem_pivoteamento(A5, b5)
T_numpy = np.linalg.solve(A5, b5)
print("T (Gauss):", T_gauss)
print("T (numpy):", T_numpy)

Matriz aumentada após passo 1:
[[ 4.   -1.    0.   15.  ]
 [ 0.    3.75 -1.   13.75]
 [ 0.   -1.    3.   10.  ]]

Matriz aumentada após passo 2:
[[ 4.         -1.          0.         15.        ]
 [ 0.          3.75       -1.         13.75      ]
 [ 0.          0.          2.73333333 13.66666667]]

T (Gauss): [5. 5. 5.]
T (numpy): [5. 5. 5.]


**Interpretação:** a matriz é diagonalmente dominante, então os pivôs não se anulam e o Gauss sem pivoteamento é estável. As três temperaturas iguais (5) indicam equilíbrio térmico uniforme na rede.

## Respostas conceituais
- **Número de soluções (ex. 3):** na forma escalonada, uma linha $0=c$ com $c\neq0$ implica nenhuma solução; menos pivôs que incógnitas, sem contradição, implica infinitas soluções; um pivô por incógnita implica solução única.
- **Limitação do método (ex. 4):** sem pivoteamento, pivô nulo interrompe o algoritmo e pivô muito pequeno amplifica erros de arredondamento.